In [ ]:
%pip install -q kagglehub "libreyolo[onnx,openvino,fast-eval]" nncf
%pip install -q --upgrade jupyter ipywidgets


In [ ]:
!git clone -q https://github.com/LuisPeregrina/gdl-atsc-anti-spillback.git gdl
%cd /content/gdl


In [ ]:
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import pathlib
from pathlib import Path
import torch
torch.serialization.add_safe_globals([pathlib._local.PosixPath])

import kagglehub
from libreyolo import LibreYOLO

print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')


In [ ]:
# Download immutable MTID source into Colab.
DATASET_NAME = "andreasmoegelmose/multiview-traffic-intersection-dataset"
source_root = Path(kagglehub.dataset_download(DATASET_NAME))
print('source_root:', source_root)


In [ ]:
# Stage 0: materialise the per-view native-COCO build (copies, not symlinks).
from tools.mtid_coco_build import CocoBuild, VIEWS

BUILD_ROOT = Path("/content/gdl/dataset_build")
view = "Drone"  # pseudo-label this view now (Drone baseline is trained)
b = CocoBuild(source_root, BUILD_ROOT, view, seed=42, materialize=True)
b.build()
_view_name = VIEWS[view][1]
build_dir = BUILD_ROOT / _view_name
print('build_dir:', build_dir)


In [ ]:
# Mount Drive and locate the trained baseline checkpoint.
from google.colab import drive
drive.mount('/content/drive')
drive_root = Path('/content/drive/MyDrive')

base = drive_root / 'gdl-atsc-anti-spillback' / 'baseline_runs'
best = sorted((base / 'mtid_drone' / 'weights').glob('best.pt'))
if not best:
    raise FileNotFoundError('best.pt not found under ' + str(base / 'mtid_drone'))
WEIGHTS = str(best[0])
print('weights:', WEIGHTS)


In [ ]:
# Stage 2: pseudo-label the sampled unannotated pool.
from tools.auto_label import pseudo_label_view

STRIDE = 30   # ~1 frame/s at 30 fps
CONF = 0.60   # min detection confidence to accept as pseudo-label

candidate, merged = pseudo_label_view(
    build_dir=build_dir,
    source_root=source_root,
    view_key=view,
    weights=WEIGHTS,
    stride=STRIDE,
    conf=CONF,
    materialize=True,
)
print('candidate:', candidate)
print('merged train:', merged)


In [ ]:
# Persist results to Drive.
import shutil
out_dir = drive_root / 'gdl-atsc-anti-spillback' / 'pseudo_labels' / view.lower()
out_dir.mkdir(parents=True, exist_ok=True)
for p in [candidate, merged]:
    shutil.copy2(p, out_dir / p.name)
    print('saved', p.name, '->', out_dir)
